In [1]:
"""Reads CSVs in the specified Drive folder; writes 0-10 scores and top three choices.
Equal attribute weights. Budget is a hard limit.
Blank scores indicate no comparable attributes. Top choices may have low scores.
Tie-breaking: score, information coverage, lower price, product ID.
No OpenAI API needed. Images are linked, not reclassified.
Drive API reference: https://developers.google.com/workspace/drive/api/guides/search-files
"""
import io
import re
import math
from pathlib import Path
import pandas as pd

FOLDER_URL = 'https://drive.google.com/drive/folders/198n-6NrRWnFP4RotCzK249ZJudn5ok67?usp=sharing'
FOLDER_ID = FOLDER_URL.split('/folders/')[1].split('?')[0]
OUTPUT_NAME = 'shampoos matches to personas.csv'
# Optional: set exact input filenames to avoid selection prompts.
SHAMPOO_FILE = None
PERSONA_FILE = None
WEIGHTS = {'gender': 1, 'hair': 1, 'scalp': 1, 'need': 1, 'scent': 1}
EXPECTED_PERSONAS = 500


def norm(x):
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')


UNKNOWN = {'', 'unknown', 'nan', 'none', 'null', 'na', 'n_a', 'not_specified',
           'unspecified', 'no_preference', 'any', 'does_not_matter', 'no_preferences'}
ALIASES = {
    'man': 'male', 'men': 'male', 'woman': 'female', 'women': 'female',
    'all_hair_types': 'all', 'all_scalp_types': 'all', 'all_types': 'all',
    'fine_flat': 'fine|flat', 'fine_thin': 'fine|thin',
    'thin_brittle': 'thin|brittle', 'dry_brittle': 'dry|brittle',
    'dry_damaged': 'dry|damaged', 'normal_to_dry': 'normal|dry',
    'normal_to_oily': 'normal|oily',
    'blonde_bleached_highlighted_silver': 'blonde|bleached|highlighted|silver|gray',
    'grey': 'gray', 'fragrancefree': 'fragrance_free', 'unscented': 'fragrance_free',
    'moisture': 'moisturizing', 'hydration': 'moisturizing', 'hydrating': 'moisturizing',
    'dandruff': 'dandruff_control', 'volume': 'fuller_appearance',
    'volumizing': 'fuller_appearance', 'volumising': 'fuller_appearance',
    'daily_cleaning': 'everyday_cleaning', 'oily_scalp': 'oily', 'dry_scalp': 'dry',
    'dry_hair': 'dry', 'oily_hair': 'oily', 'fine_hair': 'fine', 'thin_hair': 'thin',
    'damaged_hair': 'damaged', 'normal_hair': 'normal',
}


def tokens(value):
    s = norm(value)
    if s in UNKNOWN:
        return set()
    s = ALIASES.get(s, str(value).strip().lower())
    result = set()
    for part in re.split(r'[|,;/]', s):
        t = norm(part)
        mapped = ALIASES.get(t, t)
        result.update(v for v in mapped.split('|') if v not in UNKNOWN)
    return result


def number(value, label, allow_missing=True):
    if norm(value) in UNKNOWN or norm(value) in {'unlimited', 'no_limit', 'no_budget_limit'}:
        if allow_missing:
            return None
        raise ValueError(f'{label}: missing numeric value')
    s = str(value).replace('$', '').replace(',', '').strip()
    try:
        v = float(s)
    except ValueError:
        raise ValueError(f'{label}: expected a single USD amount, got {value!r}')
    if not math.isfinite(v) or v < 0:
        raise ValueError(f'{label}: amount must be finite and nonnegative')
    return v


def score_pair(person, product, pcols):
    total = earned = requested = 0.0
    details = {}
    for key, product_col in [('gender','gender_target'), ('hair','hair_type_match'),
                             ('scalp','scalp_type_match'), ('need','main_need_match'),
                             ('scent','scent_category')]:
        want = tokens(person[pcols[key]]) if pcols[key] else set()
        have = tokens(product[product_col])
        # A persona with 'all' or unisex gender has no restriction.
        if 'all' in want or (key == 'gender' and 'unisex' in want):
            want = set()
        if not want:
            details[key + '_result'] = 'no_preference'
            continue
        requested += WEIGHTS[key]
        if not have:
            details[key + '_result'] = 'unknown_product_attribute'
            continue
        total += WEIGHTS[key]
        match = bool(want & have) or 'all' in have or (key == 'gender' and 'unisex' in have)
        earned += WEIGHTS[key] * int(match)
        details[key + '_result'] = 'match' if match else 'mismatch'
    budget = number(person[pcols['budget']], 'persona budget') if pcols['budget'] else None
    price = number(product['price_usd'], 'shampoo price')
    budget_status = ('no_limit' if budget is None else 'unknown_price' if price is None
                     else 'within_budget' if price <= budget else 'over_budget')
    # Unknown product fields are ignored, per your rule, and coverage is exposed.
    score = 10 * earned / total if total else None
    coverage = total / requested if requested else 0.0
    return {**details, 'match_score': score, 'evaluated_weight': total,
            'requested_weight': requested, 'information_coverage': coverage,
            'budget_usd': budget, 'budget_status': budget_status,
            'eligible': budget_status in {'no_limit', 'within_budget'},
            'no_preferences': requested == 0}


def choose(items, title):
    if not items:
        raise ValueError(title + ': no candidates found')
    if len(items) == 1:
        return items[0]
    print('\n' + title)
    for i, item in enumerate(items, 1):
        print(i, item['name'], item.get('id',''))
    while True:
        value = input('Enter one number: ').strip()
        if value.isdigit() and 1 <= int(value) <= len(items):
            return items[int(value)-1]


def resolve_column(df, label, aliases, required=False):
    found = [c for c in df.columns if c in aliases]
    if len(found) == 1:
        return found[0]
    print(f'Choose column for {label}. Available columns: {list(df.columns)}')
    while True:
        c = norm(input('Column name' + ('' if required else ' (Enter = ignore this attribute)') + ': '))
        if c in df.columns:
            return c
        if not c and not required:
            return None


def main():
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    files, page = [], None
    while True:
        result = drive.files().list(
            q=f"'{FOLDER_ID}' in parents and trashed = false", pageSize=1000,
            pageToken=page, fields='nextPageToken,files(id,name,mimeType)',
            supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        files.extend(result.get('files', []))
        page = result.get('nextPageToken')
        if not page:
            break
    print('Files directly in folder:')
    for f in files:
        print(' -', f['name'])
    csv_files = [f for f in files if f['name'].lower().endswith('.csv') and f['name'] != OUTPUT_NAME]
    def select_input(exact, keyword, title):
        candidates = [f for f in csv_files if f['name'] == exact] if exact else [f for f in csv_files if keyword in f['name'].lower()]
        if exact and not candidates:
            raise ValueError(f'File not found: {exact}')
        return choose(candidates or csv_files, title)
    sfile = select_input(SHAMPOO_FILE, 'shampoo', 'Select shampoo catalog')
    pfile = select_input(PERSONA_FILE, 'persona', 'Select persona CSV')
    if sfile['id'] == pfile['id']:
        raise ValueError('Shampoo and persona CSVs must be different files')
    def read_csv(meta):
        buffer = io.BytesIO()
        downloader = MediaIoBaseDownload(buffer, drive.files().get_media(fileId=meta['id'], supportsAllDrives=True))
        done = False
        while not done:
            _, done = downloader.next_chunk()
        frame = pd.read_csv(io.BytesIO(buffer.getvalue()), dtype=str, keep_default_na=False, encoding='utf-8-sig')
        frame.columns = [norm(c) for c in frame.columns]
        if frame.columns.duplicated().any():
            raise ValueError('Duplicate columns after normalization')
        return frame.apply(lambda col: col.str.strip())
    shampoos, personas = read_csv(sfile), read_csv(pfile)
    required = ['product_id','image_id','brand','product_name','gender_target','hair_type_match',
                'scalp_type_match','main_need_match','price_usd','scent_category']
    missing = set(required) - set(shampoos.columns)
    if missing:
        raise ValueError(f'Shampoo columns missing: {missing}')
    specs = {
        'id': ['persona_id','person_id','participant_id','id'],
        'gender': ['gender','gender_preference','gender_target'],
        'hair': ['hair_type','hair_type_match','hair_condition','hair_target'],
        'scalp': ['scalp_type','scalp_type_match','scalp_condition','scalp_target'],
        'need': ['main_need','main_need_match','primary_need','hair_concern'],
        'scent': ['scent_preference','scent_category','preferred_scent'],
        'budget': ['budget_usd','max_budget_usd','budget','max_price_usd']}
    cols = {k:resolve_column(personas,k,v,k=='id') for k,v in specs.items()}
    for df, col in [(shampoos,'product_id'), (personas,cols['id'])]:
        if df.empty or df[col].eq('').any() or df[col].duplicated().any():
            raise ValueError(f'{col}: empty dataset, blank IDs or duplicate IDs')
    if len(personas) != EXPECTED_PERSONAS:
        print(f'NOTE: Found {len(personas)} personas; expected {EXPECTED_PERSONAS}. No rows silently removed.')
    print('Persona column mapping:', cols)
    for key in ['gender','hair','scalp','need','scent']:
        if cols[key]:
            print(key, 'persona values:', sorted(personas[cols[key]].unique()))
    # Link existing aaa/bbb/ccc images. Never guess which renamed photo matches an old filename.
    images = [f for f in files if f['mimeType'].startswith('image/')]
    image_map = {}
    for raw in shampoos['image_id'].unique():
        hits = [f for f in images if norm(Path(f['name']).stem) == norm(Path(raw).stem)]
        if len(hits) == 1:
            image_map[raw] = hits[0]
        elif images:
            image_map[raw] = choose(images, f'Which image corresponds to catalog image_id {raw!r}?')
        else:
            raise ValueError('No image files found directly in this folder')
    result_rows = []
    for person in personas.to_dict('records'):
        for product in shampoos.to_dict('records'):
            image = image_map[product['image_id']]
            row = {'persona_id': person[cols['id']], **product,
                   'image_id':Path(image['name']).stem, 'image_filename':image['name'],
                   'image_drive_url':f"https://drive.google.com/file/d/{image['id']}/view",
                   **score_pair(person, product, cols)}
            result_rows.append(row)
    out = pd.DataFrame(result_rows)
    out['rank'] = pd.Series(pd.NA, index=out.index, dtype='Int64')
    out['is_best_match'] = False
    out['match_status'] = ''
    for pid, group in out.groupby('persona_id', sort=False):
        eligible = group[group['eligible']].copy()
        if eligible.empty:
            out.loc[group.index, 'match_status'] = 'no_budget_eligible_product'
            continue
        if group['no_preferences'].all():
            out.loc[group.index,'match_status'] = 'no_preferences_all_eligible_tied'
            out.loc[eligible.index,'rank'] = 1
            out.loc[eligible.index,'is_best_match'] = True
            continue
        eligible = eligible[eligible['match_score'].notna()]
        if eligible.empty:
            out.loc[group.index,'match_status'] = 'insufficient_product_information'
            continue
        # Rank on unrounded scores. Coverage breaks score ties, not product ID or price.
        keys = sorted(set(zip(eligible.match_score,eligible.information_coverage)), reverse=True)
        ranks = {k:i+1 for i,k in enumerate(keys)}
        for idx,r in eligible.iterrows():
            rank = ranks[(r.match_score,r.information_coverage)]
            out.loc[idx,'rank'] = rank
            out.loc[idx,'is_best_match'] = rank == 1
        out.loc[group.index,'match_status'] = ('best_available_zero_match' if eligible.match_score.max()==0 else 'ranked')
    # Preserve tied ranks, but provide three deterministically ordered choices.
    out['choice_order'] = pd.Series(pd.NA, index=out.index, dtype='Int64')
    out['choice_label'] = ''
    for pid, group in out.groupby('persona_id', sort=False):
        candidates = group[group['eligible'] & (group['match_score'].notna() | group['no_preferences'])].copy()
        candidates['_price_sort'] = candidates['price_usd'].map(lambda x: number(x, 'price'))
        candidates = candidates.sort_values(
            ['match_score', 'information_coverage', '_price_sort', 'product_id'],
            ascending=[False, False, True, True], na_position='last', kind='stable')
        for position, idx in enumerate(candidates.head(3).index, 1):
            out.loc[idx, 'choice_order'] = position
            out.loc[idx, 'choice_label'] = {1:'best', 2:'second', 3:'third'}[position]
    out['is_top3'] = out['choice_order'].notna()
    out = out.sort_values(['persona_id','choice_order','rank','product_id'], kind='stable', na_position='last')
    out['match_score'] = out['match_score'].round(2)
    out['information_coverage'] = out['information_coverage'].round(4)
    assert len(out) == len(personas)*len(shampoos)
    payload = out.to_csv(index=False).encode('utf-8-sig')
    Path('/content', OUTPUT_NAME).write_bytes(payload)
    existing = [f for f in files if f['name'] == OUTPUT_NAME]
    if len(existing)>1:
        raise ValueError('Multiple output files with same name in Drive; resolve duplicates before uploading')
    media = MediaIoBaseUpload(io.BytesIO(payload), mimetype='text/csv', resumable=True)
    if existing:
        saved = drive.files().update(fileId=existing[0]['id'], media_body=media,
                                    fields='id,name',supportsAllDrives=True).execute()
    else:
        saved = drive.files().create(body={'name':OUTPUT_NAME,'parents':[FOLDER_ID]}, media_body=media,
                                    fields='id,name',supportsAllDrives=True).execute()
    print(f'Saved {len(out):,} scores ({len(personas)} personas × {len(shampoos)} shampoos).')
    print('Output:', f"https://drive.google.com/file/d/{saved['id']}/view")
    print('Filter is_top3=True for first, second and third choices; choice_order gives their order.')
    print('Blank scores mean no comparable information, not zero suitability. Budget-ineligible products are never top choices.')
    from IPython.display import display
    display(out[out.is_top3][['persona_id','choice_order','choice_label','product_id','product_name','match_score','price_usd','match_status']].head(30))
    return out


if __name__ == '__main__':
    matches = main()


Files directly in folder:
 - shampoos matches to personas.csv
 - aaa.jpg
 - shampoos latest file.csv
 - ccc.jpg
 - personas.csv
 - bbb.jpg
Persona column mapping: {'id': 'persona_id', 'gender': 'gender', 'hair': 'hair_type', 'scalp': 'scalp_type', 'need': 'main_need', 'scent': 'scent_preference', 'budget': 'budget_usd'}
gender persona values: ['female', 'male']
hair persona values: ['coily', 'curly', 'straight', 'wavy']
scalp persona values: ['dry', 'normal', 'oily']
need persona values: ['dandruff_control', 'everyday_cleaning', 'fuller_appearance', 'hair_fall_concern', 'moisturizing', 'oil_control']
scent persona values: ['citrus', 'fragrance_free', 'herbal', 'no_preference', 'woody']
Saved 12,500 scores (500 personas × 25 shampoos).
Output: https://drive.google.com/file/d/1VRJzkCAhcmSu1ZCeXJ3OTcbPfwBXUXSt/view
Filter is_top3=True for first, second and third choices; choice_order gives their order.
Blank scores mean no comparable information, not zero suitability. Budget-ineligible pr

,persona_id,choice_order,choice_label,product_id,product_name,match_score,price_usd,match_status
14,P001,1,best,SHP015,Men Old Spice Swagger 2-in-1 12.5 fl oz,6.00,6.97,ranked
15,P001,2,second,SHP016,Men Old Spice Swagger 2-in-1 20.7 fl oz,6.00,9.97,ranked
19,P001,3,third,SHP020,Men Sandalwood 2-in-1 12.5 fl oz,5.00,6.97,ranked
33,P002,1,best,SHP009,Derma Scalp Dandruff Relief Shampoo 16 fl oz,6.67,9.97,ranked
27,P002,2,second,SHP003,2-in-1 Anti-Dandruff Hydrating Shampoo & Condi...,6.00,13.97,ranked
34,P002,3,third,SHP010,Derma Scalp Density Boost Shampoo 16 fl oz,5.00,9.97,ranked
58,P003,1,best,SHP009,Derma Scalp Dandruff Relief Shampoo 16 fl oz,6.67,9.97,ranked
59,P003,2,second,SHP010,Derma Scalp Density Boost Shampoo 16 fl oz,5.00,9.97,ranked
72,P003,3,third,SHP023,EverPure Bond Repair+ Shampoo 6.8 fl oz,5.00,9.97,ranked
85,P004,1,best,SHP011,Men Old Spice Pure Sport 2-in-1 12.5 fl oz,4.00,6.97,ranked
